In [1]:
from typing import List
import torch
import numpy as np
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm
import copy, inspect
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)

DEVICE = "mps" if torch.mps.is_available() else "cpu"
#DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE = torch.float16  #if torch.cuda.is_available() else torch.float32
STEER_LAYER = 6

MODEL_NAME = "gpt2"
print("Device:", DEVICE)
print("DType:", DTYPE)

Device: mps
DType: torch.float16


# Activation Steering, pt. 2

In the first part we covered the basic idea of steering and implemented it three ways: via raw PyTorch hooks, via `nnsight`, and via `pyvene`. Did our baseline work? Yes — we saw that. Did it work well? No — we saw that too: the direction vector turned out to be inverted, the effect was moderate, and on some prompts there was no effect at all. What to do about it and how to do it in Python — that is the topic of part two.

My name is still Sabrina, and the examples in this notebook still do not reflect my personal views.

[Google Collab](https://drive.google.com/file/d/1TdwaP8WqZoYEV7dW8qV4e8NMkSfS0lct/view?usp=sharing)

## **TLDR:**

In the previous article we covered classic steering based on building a direction vector from mean differences of contrastive pairs. But the mean has problems. In this tutorial we examine those problems and open-source methods for solving them. There will be plenty of intuition and mathematics. What we will work with:

| | Method | Idea | How it differs from CAA |
|---|---|---|---|
| `repeng` | PCA on pairwise differences | finds the axis of maximum consistent spread between pairs; does not require knowing the sign of each pair in advance — but **does not provide automatic outlier protection** without additional normalization (see the correction above and the code walkthrough below) | flexibility in pair labeling, not robustness per se |
| `pyreft` | learnable low-rank intervention | the intervention is learned from data, not constructed analytically | handles non-linearity and noise (but requires enough data, otherwise overfits to the vocabulary of the pairs) |

### **Brief reminder of what we were working with.**

**Activation steering** is an inference-time intervention in a model's activations. Not fine-tuning, not prompt engineering — we literally take the activation vector during the forward pass and shift it:

$$\mathbf{h}^{(\ell)} \;\leftarrow\; \mathbf{h}^{(\ell)} + \alpha \cdot \hat{\mathbf{v}}$$

The direction vector $\hat{\mathbf{v}}$ was built using **CAA (Contrastive Activation Addition)**. We took two sets of prompts — the positive class (tolerant) and the negative (hate) — extracted the activations of the last token at the relevant layer, and computed the mean difference:

$$\hat{\mathbf{v}} = \frac{\bar{\mathbf{h}}^+ - \bar{\mathbf{h}}^-}{\|\bar{\mathbf{h}}^+ - \bar{\mathbf{h}}^-\|}$$

Note that in this formula, we normalize the entire vector to unity (rather than the contribution of each pair).

## **Preliminary definitions and intuitive meaning**

Before improving anything in any way, we need to understand our "what" and feel all the possible "hows". You can read this block before the main content, after it, or during it whenever the question "why?" comes to mind.

### **What: problems with the mean**

Since classic CAA appeals to the mean difference, let us recall the properties of the mean. The classic statistics lecture example — a situation where Bill Gates has entered your sample of population salaries.

```bash
# Monthly income, $

[2000, 3000, 3394, 2789, 2550, 5829, 2000] # before Bill, mean 3080.29
[2000, 3000, 3394, 2789, 2550, 5829, 200000] # after Bill, mean 31366.0

```
Hence the mean, and by extension any approach that uses the mean difference, is sensitive to outliers: one atypical prompt will shift the centroid vector and it can drift in the embedding space. Due to the richness of natural language, or conversely the low variability of synthetic data, atypical examples can be plentiful. This makes steering harder — in the previous tutorial, as you may recall, we did not reach the ideal. This is why the CAA approach has been improved upon.

### **What: label noise**

In our task we appeal to pairs, so noise arises (or does not arise) at the pair level. In general we know that all pairs are supposed to be about "+" and "−", but we do not rule out having pairs of "+" and "$\pm$", "$\pm$" and "−", and we may also have pairs with swapped "−"/"+" labels. These ambiguities are what we want to remove, along with the sensitivity to outliers.

Quote: it is still the principle — garbage in, garbage out. Although we set ourselves the task of minimizing noise, we are still bounded by the requirement that there is very little noise to begin with. So when steering fails — the first thing to re-check is the dataset.

### **Where we remove the problem**

Pair differences always form a matrix (call it $D$, $D \in R^{n, d_{model}}$). The mean over all vectors is what we earlier called the steering direction. The question is — how do we find that direction more robustly, in the presence of outliers and noise?

### **Geometric answers**

Our data is a cloud of vectors in $d_\text{model}$-dimensional space. Each vector $\boldsymbol{\delta}_i = \mathbf{h}^+_i - \mathbf{h}^-_i$ points roughly toward the concept, but with noise — and possibly with a flipped sign.

What does the **mean** do in this situation? It sums up points from both clusters and divides by $n$. If roughly as many pairs are flipped as are correct, the clusters cancel each other out and the mean drifts toward zero. What **does not change** when signs are mixed? Variance. And we can exploit that by using **PCA**. Look at the objective:

$$\mathbf{v}_1 = \underset{\|\mathbf{v}\|=1}{\arg\max} \sum_i (\boldsymbol{\delta}_i \cdot \mathbf{v})^2$$

This is a sum of **squared** projections. A point at $+3$ along an axis and a point at $-3$ along the same axis contribute equally — $9$ and $9$. The sign of the projection is irrelevant to the objective; only the magnitude matters. So it doesn't matter how many pairs are sign-flipped — as long as all the $\boldsymbol{\delta}_i$ (flipped or not) lie along the same axis, PCA will find that axis. The question of "who is '+' and who is '−'" is removed from the direction-finding step and deferred to the next one — determining the sign of the already-found axis, typically via a small labeled subset, after the axis has been identified.

This exact formulation — PCA on pairwise differences, with no requirement to know in advance which end of each pair is correct — is the foundation of Linear Activation Tomography from ["Representation Engineering: A Top-Down Approach to AI Transparency"](https://arxiv.org/pdf/2310.01405). There's no dedicated framework for it, but there is [source code](https://github.com/andyzoujm/representation-engineering) — and there's also an interesting derivative of that paper we're about to try (and be a little disappointed by, but then we'll try a LoReFT-based framework, and hopefully you'll be pleased by the end).

In [2]:
# same model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print(type(model))
print("n_layers:", len(model.transformer.h))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
n_layers: 12


In [3]:
# same datasets 
df = pd.read_csv("hf://datasets/SabrinaSadiekh/mixed_hate_dataset/mixed_dataset.csv", index_col=0)
df_to_not_exp = pd.read_csv("hf://datasets/SabrinaSadiekh/not_hate_dataset/not_hate_dataset.csv", index_col=0)


df.head()

,statement,is_harmfull_opposition
0,Homosexuals want to have sex with as many diff...,0
1,Violence against women is a migrant problem.,0
2,Blacks are just degenerates.,0
3,Islam is an ideology of conquest that incites ...,0
4,Mentally retarded people are uneducated and sh...,0


In [4]:
positive_prompts = (
    df[df['is_harmfull_opposition'] == 1].statement.tolist()
)
negative_prompts = (
    df[df['is_harmfull_opposition'] == 0].statement.tolist()
)

print(f"Positive (tolerant): {len(positive_prompts)}")
print(f"Negative (hate speech):   {len(negative_prompts)}")
assert len(positive_prompts) == len(negative_prompts)


Positive (tolerant): 622
Negative (hate speech):   622


### **Classic RepE vs repeng**

The `repeng` framework is a derivative of ["Representation Engineering: A Top-Down Approach to AI Transparency"](https://arxiv.org/pdf/2310.01405) — a derivative that has performed quite well. It takes the direction-extraction method from the original paper — **LAT** (Linear Artificial Tomography). The name sounds impressive and may seem intimidating, but it is simply about careful data construction and is built in three steps.

Let us look at the original algorithm:

**Step 1 — stimulus design.** The authors distinguish two types of concepts in the first step — a concept (a static object) and a function (dynamic behavior). We will only be interested in the second notion going forward, but I want to note this step.

For **concepts** (e.g., "truthfulness") — the goal is to extract declarative knowledge: the model is shown a stimulus and asked about the concept directly:

> Consider the amount of `<concept>` in the following: `<stimulus>`. The amount of `<concept>` is ___

By construction, the model will return some "measure" (not in the mathematical sense) of how much the concept is present.

For **functions** (e.g., "honesty", i.e., a behavior rather than static knowledge) — the goal is to extract procedural knowledge, so *two* templates are needed: the experimental one (asks the function to be executed) and the reference one (does not ask):

> USER: `<instruction>` `<experimental/reference prompt>`
> ASSISTANT: `<output>`

These are denoted $T_f^+$ and $T_f^-$. This is already the familiar pair structure, and going forward we will stay with the function case.

Support for the concept processing logic is not implemented in `repeng`. A function has a natural binary pair — the same instruction, two modes (execute / do not execute), which maps directly onto `(positive, negative)`. A concept has different pairs: a single template $T_c$ is applied to *different* stimuli varying in concept intensity, and the pairs for PCA and direction-finding are random pairs within one dataset, $\{A_c(i) - A_c(j)\}$, with no "plus"/"minus" role for $i$ and $j$. PCA here finds the axis of maximum spread, without relying on which element of the pair is "correct". If the concept formulation is what you need — you will have to look at the original paper's code: [github.com/andyzoujm/representation-engineering](https://github.com/andyzoujm/representation-engineering).

**Step 2 — extracting activations.** For each stimulus / function, the representation of a specific token position — by default the last token of the template — is extracted at each layer of interest. A typical dataset size is 5 to 128 pairs.

**Step 3 — building a linear model.** For function $f$, stimulus pairs yield activations on the experimental template $T_f^+$ and the reference template $T_f^-$. Even knowing the role of each pair element, the authors still randomize the sign with a $(-1)^i$ multiplier — staying true to the unsupervised formulation — and normalize each difference to unit length:

$$\boldsymbol{\delta}_i = \text{normalize}\Bigl((-1)^i\bigl(\mathbf{h}(T_f^+(q_i,a_i)) - \mathbf{h}(T_f^-(q_i,a_i))\bigr)\Bigr)$$

Then they find the first principal component of the set $\{\boldsymbol{\delta}_i\}$:

$$\mathbf{v} = \text{PC}_1\bigl(\{\boldsymbol{\delta}_i\}\bigr) = \underset{\|\mathbf{v}\|=1}{\arg\max}\sum_i (\boldsymbol{\delta}_i \cdot \mathbf{v})^2$$

The formula does not determine the sign of $\mathbf{v}$ (an eigenvector is defined up to sign) — it is found separately, after the fact, using a small labeled subset: if the projections of "+" examples turn out to be lower than those of "−" examples, $\mathbf{v}$ is simply multiplied by $-1$.


**Footnote — original details.** The paper proposes four options: (1) the reading vector added linearly — the simplest and least accurate option, since the vector does not depend on the specific input; (2) **contrast vector** — the same, but recomputed for each input at inference time via a pair of contrastive prompts; (3) **LoRRA** — low-rank adapters fine-tuned to reproduce the effect of the contrast vector without recomputation at inference time; and three ways to combine the vector with the activation — linear addition $R \pm v$, piecewise (addition conditioned on the sign of the projection), and projection (zeroing out the direction instead of amplifying it). If you want to expand your arsenal of methods beyond steering (first–partially second situation) — I encourage you to look at the original work.

We will return and first examine `repeng`-steering.

##### **Adaptation**

`repeng` is a repackaging of the LAT baseline into a pip library: the same workflow "pairs → PCA → vector → hook", but the implementation differs from the paper in four places.

**1. Pair sign is fixed.**

```python
train_strs = [s for ex in inputs for s in (ex.positive, ex.negative)]
train = h[::2] - h[1::2]  # always positive - negative
```

Unlike LAT (random order, no labels), $\boldsymbol{\delta}_i$ = positive − negative is consistent across the entire dataset. `repeng` is supervised by construction (`DatasetEntry`), just without explicit numerical labels — the "not necessarily labeled" from Step 3 no longer applies here.

**2. No normalization.** Zou et al. (Appendix C.1) explicitly state: `normalize(H(si) − H(si+1)))`. There is no such line in `extract.py` — raw differences go into `PCA(n_components=1).fit(train)` as-is. `repeng` **is more vulnerable to norm-based outliers** than the original LAT.

**3. Two methods:**

```python
# order is [positive_1, negative_1, positive_2, negative_2, ...]
if method == "pca_diff":
    train = h[::2] - h[1::2]  # pos - neg, shape (n, d) — one δ per pair

elif method == "pca_center":
    center = (h[::2] + h[1::2]) / 2  # midpoint of each pair, shape (n, d)
    train = h  # alias, not a copy — further edits mutate h too
    train[::2] -= center   # positive -= center  → +δ/2
    train[1::2] -= center  # negative -= center  → -δ/2
    # train.shape = (2n, d): all rows kept, each shifted
    # to zero relative to its pair's midpoint
```

**Toy example.**

````python
h = np.array([
    [10, 0],   # h[0] = positive_0
    [ 2, 0],   # h[1] = negative_0
    [ 8, 1],   # h[2] = positive_1
    [ 0, 1],   # h[3] = negative_1
])
````

`h[::2]` — even rows (start at 0, step 2) → **all positives**: `[[10, 0], [8, 1]]`
`h[1::2]` — odd rows (start at 1, step 2) → **all negatives**: `[[2, 0], [0, 1]]`

**`pca_diff`:**

````python
train = h[::2] - h[1::2]
# [[10-2, 0-0], [8-0, 1-1]] = [[8, 0], [8, 0]]
# shape (2, 2) — one δ per pair
````

**`pca_center`:**

````python
center = (h[::2] + h[1::2]) / 2
# [[6, 0], [4, 1]] — midpoint of each pair

train = h  # alias!
train[::2] -= center   # h[0]-c0 = [4, 0], h[2]-c1 = [4, 0]
train[1::2] -= center  # h[1]-c0 = [-4, 0], h[3]-c1 = [-4, 0]

train
# [[4, 0], [-4, 0], [4, 0], [-4, 0]]
# shape (4, 2) — every row kept, ±δ/2 pairs, symmetric around zero
````

> Note that after this branch, `h` is also mutated (same memory region as `train`) — it no longer holds the original hidden states, since `train = h` made no copy. Whether this is a bug or a feature is a question for the author, but in my view it is a bug.

### **Adaptation nuance 1: direction finding and the mean**

`repeng` offers two ways to obtain the direction, and the difference between them is not in the name but in what exactly is centered and what is ultimately maximized.

- **`pca_diff` (default).** Raw $\{\boldsymbol{\delta}_i\}$ go into PCA, and sklearn centers them internally inside `.fit()` — by subtracting the mean over the entire set. Since the sign of pairs in `repeng` is consistent (always positive − negative), this mean is close to the CAA direction. The first component here is the axis along which the distances between positive and negative *deviate* from each other most strongly.

- **`pca_center`.** Pairwise-centered data goes into PCA. For pair $i$: $\text{center}_i = \frac{\mathbf{h}^+_i + \mathbf{h}^-_i}{2}$, giving

$$\mathbf{h}^+_i - \text{center}_i = \frac{\boldsymbol{\delta}_i}{2}, \qquad \mathbf{h}^-_i - \text{center}_i = -\frac{\boldsymbol{\delta}_i}{2}$$

The sum of these two rows is $0$ for any $i$, for any $\boldsymbol{\delta}_i$: every pair cancels out algebraically, even before averaging. So the mean over the entire set $\text{mean}(\text{train}) = 0$.

Centering normally **removes the common component shared by all points** — what almost all pairs agree on — and leaves PCA to judge only individual deviations. Here this common component was already removed by the data construction itself, before any PCA. There is nothing left to subtract in `sklearn.fit()` — centering becomes a no-op (subtracting zero), and the task effectively becomes **uncentered** PCA: the first component maximizes not variance around the mean, but $\sum\|\boldsymbol{\delta}_i\|^2$ directly, i.e., simply the norm of projections.

Hence the nuance: without centering, the maximum-variance component stops distinguishing "spread around a typical value" from "simply a large magnitude for one point". One pair with an abnormally large $\|\boldsymbol{\delta}_i\|$ (e.g., Bill Gates vs. a homeless person) contributes a quadratic, unbounded amount to the sum — and can pull the direction toward itself, no matter how many other pairs point toward the true concept. This is why `pca_center` is not robust to outliers.

**4. Direction sign:**

```python
positive_smaller_mean = np.mean([projected_hiddens[i] < projected_hiddens[i+1] for i in range(0, len(inputs)*2, 2)])
positive_larger_mean = np.mean([projected_hiddens[i] > projected_hiddens[i+1] for i in range(0, len(inputs)*2, 2)])
if positive_smaller_mean > positive_larger_mean:
    directions[layer] *= -1
```

For each pair we compare `projected_hiddens[i]` (positive) and `[i+1]` (negative) **as numbers**, but what goes into the sum is not the difference but the comparison result — 0 or 1. Averaging these boolean results over all pairs gives the **fraction of pairs** with "wrong" and "correct" order; if there are more wrong ones — the sign is flipped.

**Example.** 5 pairs, `projected_hiddens` (after PCA projection) in the order `[pos, neg, pos, neg, ...]`:

```python
projected_hiddens = [
    3.0, 1.0,    # pair 0: pos > neg  → "correct"
    2.5, 0.8,    # pair 1: pos > neg  → "correct"
    1.9, 0.5,    # pair 2: pos > neg  → "correct"
    0.2, 4.7,    # pair 3: pos < neg  → "wrong" (swapped or noise)
    150.0, -80.0 # pair 4: pos > neg, but with huge norm — outlier
]
```

`positive_smaller_mean` counts the fraction of pairs where `pos < neg` → only pair 3 → `1/5 = 0.2`
`positive_larger_mean` counts the fraction of pairs where `pos > neg` → pairs 0, 1, 2, 4 → `4/5 = 0.8`

`0.2 < 0.8` → we do not flip the sign and accept the obtained vector as the direction toward positive.

Once again — pair 4, with projections `150.0` and `-80.0` (abnormally large norm), contributes **exactly the same weight** to the vote as pair 0 with projections `3.0` and `1.0` — both votes are simply "+1 to the correct majority". If instead of fraction-based voting we computed `mean(projected_hiddens[pos]) vs mean(projected_hiddens[neg])` (i.e., compared the mean *magnitudes* rather than *comparison results*), pair 4 would single-handedly dominate the outcome — `mean(pos) ≈ 31.5`, `mean(neg) ≈ -14.6` — purely due to its huge amplitude, regardless of what the other four pairs say. Fraction-based voting is protected against this: each pair gets one vote, and the extremity of its own numbers does not matter.

`ControlVector.train(...)` without an explicit `method=` uses `pca_diff` — centered but un-normalized PCA.

Training repeng means finding vectors by the described method and correcting the sign.


In [5]:
# !pip install repeng -q

### **Adaptation nuance 2: which layer the steering is applied to**

By default, `ControlVector.train(...)` without an explicit `hidden_layers` uses `range(-1, -num_hidden_layers, -1)` — i.e., **all decoder blocks except the very first** (for GPT-2: layers 1–11 out of 12). PCA is computed separately for each layer, and `ControlModel` wraps them all at once.

During generation `h ← h + α·v̂ₗ` is applied **simultaneously on 11 layers**, with a separate vector for each — not on a single layer as in the CAA baseline from part one (`STEER_LAYER = 6`).

To compare the vector extraction method in isolation, we will explicitly restrict the layer; in general, it is acceptable not to do so:

```python
ControlVector.train(ctrl_model, tokenizer, repeng_dataset, hidden_layers=[STEER_LAYER])
```
**One more detail:** `ControlModel(model, [6])` specifies only the layers for *applying* the intervention during generation — not for *training*. PCA is computed in `read_representations` via the `hidden_layers` parameter, which by default (if not passed explicitly) uses almost all layers: `range(-1, -num_hidden_layers, -1)`. `ControlVector.train(ctrl_model, tokenizer, dataset)` without `hidden_layers=[6]` ignores which `layer_ids` were used to create `ctrl_model` — and the next line (`ControlModel(model, list(repeng_vector.directions.keys()))`) recreates the wrapper on all layers for which PCA was computed.

To strictly restrict to a single layer:

```python
ctrl_model = ControlModel(model, [6])
repeng_vector = ControlVector.train(
    ctrl_model, tokenizer, repeng_dataset,
    hidden_layers=[6],   # without this — defaults to all layers
)
trained_layer_ids = list(repeng_vector.directions.keys())  # now [6]
ctrl_model = ControlModel(model, trained_layer_ids)
```


In [6]:
# repeng uses np.float_, removed in NumPy 2.0 — patch before import
if not hasattr(np, "float_"):
    np.float_ = np.float64

REPENG_OK = False


try:
    from repeng import ControlVector, ControlModel, DatasetEntry
except ImportError:
    print("repeng not installed: pip install repeng")
except Exception as e:
    print(f"repeng: error — {type(e).__name__}: {e}")
    

repeng_dataset = [
        DatasetEntry(positive=p, negative=n)
        for p, n in zip(positive_prompts, negative_prompts)
    ]

n_layers = len(model.transformer.h)
ctrl_model = ControlModel(model, [6])

repeng_vector = ControlVector.train(ctrl_model, tokenizer, repeng_dataset, hidden_layers=[6])

# repeng numbers directions with an offset (hidden_states[0] = embedding, skipped).
# Recreate ControlModel with the layers for which directions were actually computed.
trained_layer_ids = list(repeng_vector.directions.keys())
ctrl_model = ControlModel(model, trained_layer_ids)

REPENG_OK = True
print(f"repeng: trained on {len(repeng_dataset)} pairs, layers: {trained_layer_ids}")


100%|██████████| 1/1 [00:00<00:00, 191.85it/s]

repeng: trained on 622 pairs, layers: [6]



/Users/sabrinasadieh/Code/XAI-open_materials/.venv/lib/python3.12/site-packages/repeng/control.py:37: UserWarning: Trying to rewrap a wrapped model! Probably not what you want! Try calling .unwrap first.
  warnings.warn(


In [7]:
eval_prompt = "Are blacks dangerous? The answer is:"

In [8]:
if REPENG_OK:
    def repeng_generate(prompt: str, coeff: float, max_new_tokens: int = 15) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        ctrl_model.set_control(repeng_vector, coeff=coeff)
        try:
            with torch.no_grad():
                out = ctrl_model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            return tokenizer.decode(out[0], skip_special_tokens=True)
        finally:
            # Guarantee reset even on exception — otherwise the model stays steered
            ctrl_model.reset()

    print("=== repeng  coeff=0  (baseline) ===")
    print(repeng_generate(eval_prompt, coeff=0.0))

    print("\n=== repeng  coeff=+2  (→ tolerant) ===")
    print(repeng_generate(eval_prompt, coeff=16.0))

    print("\n=== repeng  coeff=-2  (→ hate) ===")
    print(repeng_generate(eval_prompt, coeff=-16.0))


=== repeng  coeff=0  (baseline) ===
Are blacks dangerous? The answer is: they're not.

The study, published in the journal Science,

=== repeng  coeff=+2  (→ tolerant) ===
Are blacks dangerous? The answer is: they're not.

The problem is that blacks are not dangerous.

=== repeng  coeff=-2  (→ hate) ===
Are blacks dangerous? The answer is: Yes.

The study, published in the journal Science, found that


What we solved and what we did not.

The PCA objective function is **quadratic** in $\boldsymbol{\delta}_i$: the contribution of one pair grows as $\|\boldsymbol{\delta}_i\|^2$. For the mean, the contribution of a pair is linear and hard-capped — $\boldsymbol{\delta}_i / n$. So for a single extreme outlier the situation is exactly the opposite of what one might expect: for the mean, the influence of the outlier grows linearly with its magnitude; for PCA — quadratically. One sufficiently long $\boldsymbol{\delta}_i$ can determine the top eigenvector almost single-handedly, "overpowering" the combined contributions of all other pairs — Bill Gates will not just shift the mean, he will also drag the principal axis along with him.

PCA can be more useful than the mean — but within the (reasonable) limits of the method: for example, if each $\boldsymbol{\delta}_i$ is pre-normalized to unit length (then the quadratic term cannot blow up from a single long pair). This is exactly what the RepE authors do (Zou et al., Appendix C.1): `normalize(H(si) − H(si+1)))` before PCA.

**In `repeng` this protection is absent in both `pca_diff` and `pca_center`**. So the outlier (noise) problem is fully present here. The library gives us a different way to find the direction vector. And if you truly want to solve the problem using PCA, only the theory above will help.


## **Forget your geometry: let's learn**

The second framework in focus — `pyreft` (from Representation Fine-Tuning, ReFT). It implements a completely different approach: **the intervention is learned from data**, not constructed analytically.

**ReFT is a family of methods, not a single algorithm.** The original paper ([ReFT: Representation Finetuning for Language Models](https://arxiv.org/pdf/2404.03592), Wu et al., NeurIPS 2024) describes two concrete members:

- **LoReFT** (Low-rank Linear Subspace ReFT) — intervention in a low-rank linear subspace.
- **DiReFT** — a history that is more optimal with the main motivation — to push the intervention not into the weights, but directly into the model's residual stream (its representations).

### **LoReFT mathematics**

LoReFT learns an intervention of the form:

$$\mathbf{h} \;\leftarrow\; \mathbf{h} + \mathbf{R}^\top \bigl(\mathbf{W}\mathbf{h} + \mathbf{b} \;-\; \mathbf{R}\mathbf{h}\bigr)$$

Let us break this down — it is no scarier than LAT:

- $\mathbf{R} \in \mathbb{R}^{r \times d}$ — **low-rank projector** ($r \ll d$, e.g. $r=4$ with $d=768$). Its rows form an orthonormal basis for a small $r$-dimensional subspace inside the activation space.
- $\mathbf{R}\mathbf{h} \in \mathbb{R}^r$ — the projection of $\mathbf{h}$ onto this subspace: "the coordinates of $\mathbf{h}$ inside it".
- $\mathbf{W}\mathbf{h} + \mathbf{b} \in \mathbb{R}^r$ — the **desired** coordinates in the same subspace.
- $(\mathbf{W}\mathbf{h} + \mathbf{b} - \mathbf{R}\mathbf{h})$ — the error between the current projection and the desired one.
- $\mathbf{R}^\top(\ldots)$ — "lifts" the correction back into the $d$-dimensional space and adds it to $\mathbf{h}$.

**Informally:** we shift $\mathbf{h}$ only inside a small $r$-dimensional "corridor", leaving the remaining $d-r$ dimensions untouched.

### **DiReFT mathematics**

DiReFT removes a couple of things from LoReFT.

$$\Phi_{\text{DiReFT}}(\mathbf{h}) = \mathbf{h} + \mathbf{W}_2^\top \bigl(\mathbf{W}_1\mathbf{h} + \mathbf{b}\bigr)$$

**What / why:**

1. **Difference operation.** In LoReFT the correction is the difference between the *desired* projection ($\mathbf{W}\mathbf{h}+\mathbf{b}$) and the *current* projection ($\mathbf{R}\mathbf{h}$): the intervention knows where $\mathbf{h}$ already is in the subspace and moves it by exactly the missing difference. In DiReFT there is no such comparison — $\mathbf{W}_1\mathbf{h}+\mathbf{b}$ is added in full, without subtracting the current state.
2. **Orthogonality.** $\mathbf{R}$ in LoReFT is a matrix with orthonormal rows (this guarantees that the subspace behaves as a "clean" $r$-dimensional slice without metric distortions). In DiReFT $\mathbf{W}_1$ and $\mathbf{W}_2$ are simply two independent low-rank matrices, without this constraint.

The DiReFT equation is structurally identical to LoRA. The difference from ordinary LoRA is only *where* the adapter is applied — not to the layer weights, but directly to the activation vector.

**Trade-off and place.** For DiReFT — fewer constraints — faster training (no need to maintain orthogonality of $\mathbf{R}$ at every step, no need to compute $\mathbf{R}\mathbf{h}$ separately). But the paper explicitly says this is an ablation, not an improvement: DiReFT *"trades some performance for increased efficiency"*. The number of trainable parameters is the same for both methods ($2rd+r$), so the difference is not in size but in formulation.

### **Training objective**

The paper considers two settings in parallel. A model with ReFT intervention $\Phi$ and trainable parameters $\varphi$ is denoted $p_\Phi(\cdot)$.

**Generation** (decoder-only / encoder-decoder LM): given a prompt $x=(x_1,\ldots,x_n)$, we need to predict $y=(y_1,\ldots,y_m)$ — the usual cross-entropy with teacher forcing over all output positions, the same loss as when training the LM itself, but the gradient flows into $\varphi$ ($\mathbf{R}, \mathbf{W}, \mathbf{b}$) with the base model frozen.

**Classification** (encoder-only): a head $H_\theta(\cdot)$ on top of the CLS-token representation of the final layer, minimizing the cross-entropy of the target class $y$ given input $x$.

In both settings the intervention $\Phi$ is embedded in the forward pass at specific positions/layers and trained with standard gradient descent — unlike LAT/`repeng`, where the direction is found analytically (PCA), without a single backprop step.

### **Why this is good**

- **We do not change the model weights** — only the intervention parameters ($\mathbf{R}, \mathbf{W}, \mathbf{b}$)
- **Few trainable parameters**: $2rd + r$ total; at $r=4, d=768$ — around 6K instead of 124M GPT-2 weights
- **Works where CAA fails**: if the concept is non-linear or noisy, the learned intervention will find it better than mean difference


### **Why this is not ideal**

In CAA and repeng there is an explicit discrete step: extract activations $\mathbf{h}^+$ and $\mathbf{h}^-$, compute the difference $\boldsymbol{\delta}_i$, find the direction (mean or PC1). The direction is an object you can pull out and inspect.

In LoReFT there is no such step at all. The training data is pairs of **texts**, not activations:

```python
data_module = pyreft.make_last_position_supervised_data_module(
    tokenizer=tokenizer, model=reft_model,
    inputs=negative_prompts[:],   # x — hate text
    outputs=positive_prompts[:],  # y — tolerant text
)
```

`make_last_position_supervised_data_module` ([`pyreft/dataset.py`](https://github.com/stanfordnlp/pyreft/blob/main/pyreft/dataset.py)) concatenates $x$ and $y$ into one sequence and tokenizes it as a whole; the labels of prompt tokens are masked with `IGNORE_INDEX`, so the loss is computed only over the tokens of $y$. The intervention is physically applied **at one point** — at the last token of the prompt (`intervention_locations = [[base_prompt_length - 1]]`), but thanks to causal attention this single modified representation is "visible" to all subsequent tokens during their generation — so one position is enough to influence the entire completion. For intervention at multiple positions the library provides a separate function `make_multiple_position_supervised_data_module`. Beyond that, this training is no different from ordinary fine-tuning: the authors write directly in the README — *"Now, you could train ReFT just like any next token prediction tasks!"* — teacher forcing, cross-entropy over tokens of $y$, backprop into $\mathbf{R}, \mathbf{W}, \mathbf{b}$ with the base model frozen (the Training objective discussed above).

**Key difference:** "contrastiveness" here is not a mathematical object (a difference of vectors), but a property of the *training data* — the fact that $x$ and $y$ systematically differ along the desired dimension (hate vs. tolerant) causes the gradient to repeatedly push $\mathbf{R}, \mathbf{W}, \mathbf{b}$ in the same direction. The "direction" in LoReFT is the *behavior* of the trained module: what it has learned is spread across three matrices and can only be recovered empirically — by running different $\mathbf{h}$ through $\Phi_{\text{LoReFT}}$ and observing where it shifts them, rather than reading a single vector from the weights.


In [9]:
# !pip install pyreft -q

In [10]:
PYREFT_OK = False

try:
    import copy, inspect
    import pyreft
    import pyvene as pv
    import transformers

    reft_config = pyreft.ReftConfig(representations=[
        pv.RepresentationConfig(
            layer=STEER_LAYER,
            component="block_output",
            low_rank_dimension=4,
            intervention_type=pyreft.LoreftIntervention,
        )
    ])

    # LoreftIntervention uses linalg_householder_product — not supported on MPS (i have mac, sorry)).
    # Create a CPU copy of the model specifically for pyreft; leave the original untouched.
    _reft_base = copy.deepcopy(model).cpu().float()
    reft_model = pyreft.get_reft_model(_reft_base, reft_config)
    reft_model.float()  # LoreftIntervention may be created in BFloat16 — align dtype
    reft_model.print_trainable_parameters()

    data_module = pyreft.make_last_position_supervised_data_module(
        tokenizer=tokenizer,
        model=reft_model,
        inputs=positive_prompts[:],
        outputs=negative_prompts[:], # order matters — we want hateful outputs
    )

    # transformers >= 5.x passes num_items_in_batch to compute_loss — patch it
    _orig_compute_loss = pyreft.ReftTrainer.compute_loss
    def _compute_loss_compat(self, model, inputs, return_outputs=False, **kwargs):
        return _orig_compute_loss(self, model, inputs, return_outputs)
    pyreft.ReftTrainer.compute_loss = _compute_loss_compat

    tok_key = (
        "processing_class"
        if "processing_class" in inspect.signature(pyreft.ReftTrainer).parameters
        else "tokenizer"
    )

    training_args = transformers.TrainingArguments(
        num_train_epochs=10,
        output_dir="/tmp/pyreft_hate",
        learning_rate=5e-3,
        per_device_train_batch_size=4,
        logging_steps=10,
        report_to="none",
        remove_unused_columns=False,   # intervention_locations must reach compute_loss
        use_cpu=True,                   # explicitly CPU — MPS does not support householder_product
    )
    trainer = pyreft.ReftTrainer(
        model=reft_model,
        args=training_args,
        **{tok_key: tokenizer},
        **data_module,
    )
    _ = trainer.train()
    reft_model.eval()
    PYREFT_OK = True
    print("pyreft: training done")

except ImportError:
    print("pyreft not installed: pip install pyreft")
except Exception as e:
    print(f"pyreft: error — {type(e).__name__}: {e}")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 50256, 'bos_token_id': 50256, 'pad_token_id': 50256}.


trainable intervention params: 6,148 || trainable model params: 0
model params: 124,439,808 || trainable%: 0.004940541213306919


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,2.859969
20,2.231573
30,2.204397
40,1.886274
50,1.858578
60,2.004977
70,1.718389
80,1.668584
90,1.608713
100,2.063081


Directory /tmp/pyreft_hate/checkpoint-500/intervenable_model already exists and contains files. Skipping save to prevent overwriting existing model.
Directory /tmp/pyreft_hate/checkpoint-1000/intervenable_model already exists and contains files. Skipping save to prevent overwriting existing model.
Directory /tmp/pyreft_hate/checkpoint-1500/intervenable_model already exists and contains files. Skipping save to prevent overwriting existing model.
Directory /tmp/pyreft_hate/checkpoint-1560/intervenable_model already exists and contains files. Skipping save to prevent overwriting existing model.


pyreft: training done


In [11]:
if PYREFT_OK:
    def pyreft_generate(prompt: str, max_new_tokens: int = 60) -> str:
        """Greedy generation with the trained ReFT intervention.

        reft_model.generate() uses the nnsight backend of pyvene — unstable on MPS.
        We use a manual loop via reft_model() (forward pass), as in pyvene_generate.
        Inputs are kept on CPU — reft_model was trained on CPU.
        """
        inputs = tokenizer(prompt, return_tensors="pt")  # CPU, no .to(DEVICE)
        generated = inputs.input_ids

        with torch.no_grad():
            for _ in range(max_new_tokens):
                seq_len = generated.shape[1]

                _, out = reft_model(
                    {"input_ids": generated},
                    unit_locations={"sources->base": (None, [[[seq_len - 1]]])},
                )

                # pyvene returns (None, steered_output) — take the second element
                next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=1)

                if next_token.item() == tokenizer.eos_token_id:
                    break

        return tokenizer.decode(generated[0], skip_special_tokens=True)

    print("=== pyreft (trained ReFT intervention) ===")
    print(pyreft_generate(eval_prompt))


=== pyreft (trained ReFT intervention) ===
Are blacks dangerous? The answer is:Black people are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.Blacks are dangerous.


Pyreft ran — and produced a loop. This is normal: GPT-2 is small and struggles. The key takeaway for us is whether the learned intervention can push the model in the desired direction at all. It can — hate output was produced.

## Summary

Wow, if you are reading these lines — thank you. We have covered a lot of ground. A summary table for clarity:

| | CAA | repeng | pyreft |
|---|---|---|---|
| How the direction is found | mean(pos) − mean(neg) | PCA on $\boldsymbol{\delta}_i$ | gradient descent |
| Pair sign must be known in advance | yes | no* | no |
| Sensitive to norm-based outliers | linearly | quadratically** | — |
| Layer coverage | 1 | configurable | configurable |
| Non-linear concept | no | no | yes |
| Requires training | no | no | yes |
| Direction can be extracted and inspected | yes | yes | no*** |

**Notes:**

- `*` PCA works on squared projections — the sign does not matter. `repeng` fixes the sign through the `DatasetEntry` construction, but the original LAT from the paper does not do this and you can go to it directly.
- `**` Without normalizing $\boldsymbol{\delta}_i$ before PCA — an outlier's contribution grows quadratically. The original LAT normalizes; `repeng` does not.
- `***` The LoReFT "direction" is spread across $\mathbf{R}, \mathbf{W}, \mathbf{b}$ and can only be recovered by running different $\mathbf{h}$ through $\Phi$.

And a few practical bullet points, so you don't have to keep a lot of text in your head:

- If the dataset is clean and large — CAA works and you don't need anything more complex.
- If the sign of pairs is unreliable (noisy labels, minimal pairs) — LAT (but not `repeng` as-is) removes the sign problem, but not the norm-outlier problem; $\boldsymbol{\delta}_i$ normalization is needed.
- If the concept is non-linear or data is scarce — pyreft. But you need a sufficiently large model and enough data, otherwise you get a loop.

If you enjoyed this, join [Just Data Blog](https://t.me/jdata_blog) — I will become a followed channel and will be happy that what I produce brings more interesting things into the world.

See you next time!
